### Machine Performance Monitor

Supports monitoring a remote machine, a VM inside a remote machine, or the local machine.

* **Configuration**: Edit the settings in the first code cell.
* **Remote/Local**: Use `MONITOR_REMOTE` to switch between remote and local monitoring.
* **VM/Host**: Use `MONITOR_REMOTE_VM` to switch between monitoring the VM and the jump host.
* **SSH Config**: For remote monitoring, set up your SSH credentials in the `sshConfig` file.
* **Metrics**: Toggle the boolean flags (`MONITOR_CPU`, `MONITOR_MEM`, etc.) to enable or disable specific metrics.

In [ ]:
# ==================== Configuration ====================
config = {
    'MONITOR_REMOTE': False,     # True to monitor remote, False to monitor local
    'MONITOR_REMOTE_JUMP': False,  # True for using jump function(vm in remote). False to close jump function(monitoring directly on remote)
    # --- monitoring settings ---
    'INTERVAL': 5,              # Sampling interval (seconds)
    'DURATION': 30,             # Total duration (seconds), set to None for infinite execution
    # --- Switches for metrics ---
    'MONITOR_CPU': True, 
    'MONITOR_MEM': True,
    'MONITOR_DISK_USAGE': True,
    'MONITOR_DISK_IO': False,    # Requires iostat (sysstat package)
    'MONITOR_NETWORK': True,
    'MONITOR_LOAD': True,
    'MONITOR_PROCESSES': True,
    # --- Device names ---
    'NET_INTERFACE': 'eth0',      # e.g., eth0, ens33
    'DISK_DEVICE': 'sda'          # e.g., sda, vda
}

In [ ]:
from monitoring_logic import setup_client, MetricsCollector, draw_plots
import time

# ==================== Monitoring Loop ====================
client, HOST_ALIAS = setup_client(config['MONITOR_REMOTE'], config['MONITOR_REMOTE_JUMP'], ssh_config_path="sshConfig")
collector = MetricsCollector(client, config)

times = []
data = {
    'cpu': [], 'mem': [], 'disk_usage': [], 'disk_io': [],
    'net_sent': [], 'net_recv': [], 'load1': [], 'load5': [], 'load15': [],
    'processes': []
}

start_time = time.time()
print(f"Starting to monitor {HOST_ALIAS}, updating every {config['INTERVAL']} seconds...")
print("Press the ■ stop button in the Jupyter toolbar to stop.\n")

try:
    while True:
        elapsed = time.time() - start_time
        if config['DURATION'] and elapsed >= config['DURATION']:
            print("\nMonitoring duration reached, stopping...")
            break
        
        times.append(elapsed)
        collector.collect_all_metrics(data)
        draw_plots(times, data, config, HOST_ALIAS)
        
        time.sleep(config['INTERVAL'])
except KeyboardInterrupt:
    print("\nMonitoring interrupted by user.")
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    client.close()
    print("Monitoring finished, connection closed.")